# Fine-tune VLA-Adapter on RobotArmLearning

This notebook downloads the public RobotArmLearning demonstrations from Hugging Face, regenerates the shoulder and wrist observations in headless MuJoCo, converts them to RLDS, and fully fine-tunes the Qwen2.5-0.5B VLA-Adapter model.

Before running, select **Runtime → Change runtime type** and pick an **A100 (recommended)**. VLA-Adapter trains in bfloat16 throughout, which needs an Ampere-or-newer GPU; the free-tier T4 is Turing and PyTorch will refuse the first autocast. The preflight cell checks this. All vision, language, projection, and action-query parameters are trainable, along with the action head and proprio projector. Full fine-tuning needs more GPU memory than LoRA; an L4 may require additional memory tuning. Allow at least 50 GiB of free runtime disk, plus Drive space for the full model, optimizer state, and exported archive. Preview `sim.mp4` files are intentionally not downloaded; the compact state trajectories are sufficient to regenerate the training images.

In [ ]:
# User settings
DATASET_REPO = "FoxNerdSaysMoo/robot-arm-learning-data"
DATASET_REVISION = "main"  # Replace with a commit SHA to freeze the dataset.
CODE_REPO = "https://github.com/zebulontaylor/RobotArmTraining.git"
VLA_REPO = "https://github.com/OpenHelix-Team/VLA-Adapter.git"
VLA_COMMIT = "23fa0c9c159e2aa04341cdd3e924f44061311060"
MODEL_REPO = "Stanford-ILIAD/prism-qwen25-extra-dinosiglip-224px-0_5b"
INSTRUCTION = "stack the three colored cubes"
SAMPLE_HZ = 10.0
MAX_STEPS = 10_000       # Use 50 first for a quick end-to-end test.
SAVE_FREQ = 1_000
VAL_FRACTION = 0.1       # Hold out complete episodes, never individual frames.
VAL_SPLIT_SEED = 20260920
VAL_FREQ = 250           # Validate every 250 optimizer steps.
VAL_TIME_LIMIT = 60      # Bound validation overhead between training windows.
BATCH_SIZE = 1           # Conservative starting point for full fine-tuning.
GRAD_ACCUM_STEPS = 8     # Effective batch size: 1 x 8 = 8.
LEARNING_RATE = 2e-5     # Full-model learning rate; tune against held-out loss.
SAVE_TO_DRIVE = True     # Checkpoints must outlive the runtime to be resumable.
DRIVE_OUTPUT = "/content/drive/MyDrive/robot-arm-learning-vla-outputs"
SHUTDOWN_WHEN_DONE = False  # True releases the Colab runtime after the export cell.
RESUME_RUN_ID = ""       # A full-finetuning RUN_ID; old LoRA runs cannot be resumed.
RESUME_LEARNING_RATE = None  # For example, 1e-5 overrides the checkpoint LR on resume.
WANDB_ENTITY = ""        # Your W&B user or team. Empty keeps logging offline.
WANDB_PROJECT = "robot-arm-learning-panthera"


In [ ]:
# GPU and disk preflight
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU detected. Enable a GPU runtime before continuing.")
subprocess.run(["nvidia-smi"], check=True)

# VLA-Adapter casts weights, inputs and autocast to bfloat16 in a dozen places
# and carries no GradScaler, so a pre-Ampere GPU cannot run it unconverted.
try:
    import torch
    capability = torch.cuda.get_device_capability()
except Exception:
    capability = None
if capability is None:
    print("Could not read compute capability; skipping the bfloat16 check.")
elif capability[0] < 8:
    raise RuntimeError(
        f"{torch.cuda.get_device_name(0)} is compute capability {capability[0]}.{capability[1]}, "
        "which has no bfloat16 support. Choose an L4 or A100 runtime.")
else:
    print(f"Compute capability {capability[0]}.{capability[1]}: bfloat16 supported.")

free_gb = shutil.disk_usage("/content").free / 2**30
print(f"Free runtime disk: {free_gb:.1f} GiB")
if free_gb < 50:
    raise RuntimeError("At least 50 GiB of free runtime disk is recommended.")


In [ ]:
# Mount Drive first: the permission dialog blocks until it is clicked, and
# waiting to ask until the training cell would stall an unattended Run All
# behind an hour of setup. Mounting is idempotent, so the training cell's own
# call is a no-op after this.
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Grant the Drive prompt now; the rest of the notebook runs unattended.")


## 1. Fetch code and create an isolated Python 3.10 environment

VLA-Adapter pins PyTorch 2.2 and TensorFlow 2.15. The separate environment avoids conflicts with Colab's preinstalled packages. Long setup output is normal.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path("/content")
CODE_DIR = ROOT / "RobotArmLearning"
VLA_DIR = CODE_DIR / "VLA-Adapter"
VENV = ROOT / "vla-env"
PYTHON = str(VENV / "bin/python")

def run(command, **kwargs):
    """Run a subprocess, streaming its output into the notebook.

    A child process writes to file descriptor 1, which bypasses the
    `sys.stdout` that ipykernel replaces, so `subprocess.run` output is
    invisible here. Reading the pipe ourselves keeps progress lines -- and
    failures -- in the cell where they can be seen.
    """
    print("+", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               **kwargs)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

if not CODE_DIR.exists():
    run(["git", "clone", "--depth=1", CODE_REPO, CODE_DIR])
if not VLA_DIR.exists():
    run(["git", "clone", VLA_REPO, VLA_DIR])
run(["git", "checkout", VLA_COMMIT], cwd=VLA_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
run(["uv", "python", "install", "3.10"])
if not VENV.exists():
    run(["uv", "venv", VENV, "--python", "3.10"])
# protobuf has to land on 4.x: TensorFlow 2.15 caps it below 5, and wandb
# 0.26+ dropped the generated bindings for protobuf 3. Nothing here pins it,
# and current tensorflow_metadata resolves to a release whose *_pb2.py modules
# import google.protobuf.runtime_version -- protobuf 5.27 and later only.
# 1.13.1 is the newest that both predates that and permits 4.x; the tfmd
# releases in between cap protobuf below 4.21 and so break wandb instead.
run(["uv", "pip", "install", "--python", PYTHON, "-e", VLA_DIR,
     "tensorflow-metadata==1.13.1", "protobuf==4.25.9",
     "mujoco", "opencv-python-headless", "scipy"])
run([PYTHON, "-c", "import torch, tensorflow as tf, mujoco; "
     "print('torch', torch.__version__, 'tensorflow', tf.__version__, 'mujoco', mujoco.__version__)"])


## 2. Register RobotArmLearning with VLA-Adapter

The upstream trainer only knows its built-in Open-X datasets. This idempotent patch registers RobotArmLearning's two cameras, 8-D proprioceptive state, and 7-D end-effector action. It also enables full-model training and checkpoint resume, optional gradient checkpointing, prevents duplicate checkpoint writes during gradient accumulation, and makes the FlashAttention 2 request conditional on the GPU actually supporting it.

In [ ]:
def insert_after(path, anchor, addition, marker):
    text = path.read_text()
    if marker not in text:
        if anchor not in text:
            raise RuntimeError(f"Patch anchor not found in {path}")
        path.write_text(text.replace(anchor, anchor + addition, 1))

oxe = VLA_DIR / "prismatic/vla/datasets/rlds/oxe"
insert_after(
    oxe / "configs.py",
    "OXE_DATASET_CONFIGS = {\n",
    '    "robot_arm_learning_panthera": {\n'
    '        "image_obs_keys": {"primary": "image", "secondary": None, "wrist": "wrist_image"},\n'
    '        "depth_obs_keys": {"primary": None, "secondary": None, "wrist": None},\n'
    '        "state_obs_keys": ["state"],\n'
    '        "state_encoding": StateEncoding.POS_EULER,\n'
    '        "action_encoding": ActionEncoding.EEF_POS,\n'
    '    },\n',
    '"robot_arm_learning_panthera"',
)
insert_after(
    oxe / "mixtures.py",
    "OXE_NAMED_MIXTURES: Dict[str, List[Tuple[str, float]]] = {\n",
    '    "robot_arm_learning_panthera": [("robot_arm_learning_panthera", 1.0)],\n',
    '"robot_arm_learning_panthera"',
)
insert_after(
    oxe / "transforms.py",
    "OXE_STANDARDIZATION_TRANSFORMS = {\n",
    '    "robot_arm_learning_panthera": lambda trajectory: trajectory,\n',
    '"robot_arm_learning_panthera"',
)

# The Qwen2.5 backbone asks for FlashAttention 2 unconditionally. VLA-Adapter
# leaves flash_attn to a manual post-install step, and it needs an Ampere or
# newer GPU regardless -- a T4 is Turing and cannot run it at all. Fall back to
# PyTorch SDPA unless both the hardware and the package are present.
base_llm = VLA_DIR / "prismatic/models/backbones/llm/base_llm.py"
insert_after(
    base_llm,
    "        return self.tokenizer.pad_token_id\n\n\n",
    "def flash_attention_2_available() -> bool:\n"
    '    """FlashAttention 2 needs an Ampere-or-newer GPU and the flash_attn package."""\n'
    "    import importlib.util\n"
    "    if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8:\n"
    "        return False\n"
    '    return importlib.util.find_spec("flash_attn") is not None\n\n\n',
    "def flash_attention_2_available",
)
attention = base_llm.read_text()
old_attn = "use_flash_attention_2=use_flash_attention_2 if not self.inference_mode else False,"
new_attn = ("use_flash_attention_2=use_flash_attention_2 and flash_attention_2_available()\n"
            "                if not self.inference_mode else False,")
if old_attn in attention:
    base_llm.write_text(attention.replace(old_attn, new_attn, 1))

finetune = VLA_DIR / "vla-scripts/finetune.py"
insert_after(
    finetune,
    "    use_pro_version: bool = True                             # the version number\n",
    "    use_gradient_checkpointing: bool = False\n",
    "use_gradient_checkpointing: bool",
)
checkpoint_anchor = "    # FiLM setup\n"
checkpoint_block = (
    "    if cfg.use_gradient_checkpointing:\n"
    "        llm = (vla.base_model.model if cfg.use_lora else vla).language_model\n"
    "        llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": False})\n"
    "        llm.config.use_cache = False\n"
    "        print(\"Gradient checkpointing enabled on language_model\")\n\n"
)
text = finetune.read_text()
if "Gradient checkpointing enabled on language_model" not in text:
    if checkpoint_anchor not in text:
        raise RuntimeError("Gradient-checkpointing patch anchor not found")
    text = text.replace(checkpoint_anchor, checkpoint_block + checkpoint_anchor, 1)
old_save = "if gradient_step_idx > 0 and log_step % cfg.save_freq == 0:"
new_save = ("if gradient_step_idx > 0 and log_step % cfg.save_freq == 0 "
            "and (batch_idx + 1) % cfg.grad_accumulation_steps == 0:")
if old_save in text:
    text = text.replace(old_save, new_save, 1)
finetune.write_text(text)

# Preserve physical zero when percentile-normalizing relative EEF actions.
rlds_dataset = VLA_DIR / "prismatic/vla/datasets/rlds/dataset.py"
statistics_text = rlds_dataset.read_text()
statistics_old = (
    '        full_dataset = dl.DLataset.from_rlds(\n'
    '            builder, split="all", shuffle=False, num_parallel_reads=num_parallel_reads\n'
    '        ).traj_map(restructure, num_parallel_calls)\n'
)
statistics_new = (
    '        statistics_split = ("train" if name == "robot_arm_learning_panthera" '
    'and "val" in builder.info.splits else "all")\n'
    '        full_dataset = dl.DLataset.from_rlds(\n'
    '            builder, split=statistics_split, shuffle=False, num_parallel_reads=num_parallel_reads\n'
    '        ).traj_map(restructure, num_parallel_calls)\n'
)
if "statistics_split =" not in statistics_text:
    if statistics_old not in statistics_text:
        raise RuntimeError("Training-only statistics patch anchor not found")
    statistics_text = statistics_text.replace(statistics_old, statistics_new, 1)
    # Include the selected split in the statistics cache key so an older
    # all-data cache cannot leak validation distribution information.
    statistics_text = statistics_text.replace(
        '                str(builder.info),\n',
        '                str(builder.info),\n                statistics_split,\n',
        1,
    )
    rlds_dataset.write_text(statistics_text)
insert_after(
    rlds_dataset,
    "    dataset_statistics = tree_map(np.array, dataset_statistics)\n",
    '\n    if name == "robot_arm_learning_panthera":\n'
    '        action_stats = dataset_statistics["action"]\n'
    '        delta_scale = np.maximum(\n'
    '            np.abs(action_stats["q01"][:6]),\n'
    '            np.abs(action_stats["q99"][:6]),\n'
    '        )\n'
    '        action_stats["q01"][:6] = -delta_scale\n'
    '        action_stats["q99"][:6] = delta_scale\n',
    'if name == "robot_arm_learning_panthera"',
)
insert_after(
    finetune,
    "    use_gradient_checkpointing: bool = False\n",
    "    balance_z_loss: bool = False\n"
    "    z_neutral_threshold: float = 0.045\n"
    "    z_down_weight: float = 1.0\n"
    "    z_neutral_weight: float = 0.7\n"
    "    z_up_weight: float = 1.8\n",
    "balance_z_loss: bool",
)
# The trainer hardcodes `mode="offline"` and never passes --wandb_entity to
# wandb.init, so neither the flag nor WANDB_MODE can reach it. Honour both.
text = finetune.read_text()
old_init = 'wandb.init(project=cfg.wandb_project, name=f"ft+{run_id}", mode="offline")'
new_init = ('wandb.init(entity=cfg.wandb_entity or None, project=cfg.wandb_project,\n'
            '                   name=f"ft+{run_id}", mode=os.environ.get("WANDB_MODE", "offline"))')
if old_init in text:
    finetune.write_text(text.replace(old_init, new_init, 1))

# Make the upstream validation path deterministic and compatible with the
# project-specific forward-pass configuration.
text = finetune.read_text()
old_val_dataset = '            image_aug=cfg.image_aug,\n            train=False,\n'
new_val_dataset = '            image_aug=False,\n            train=False,\n'
if old_val_dataset in text:
    text = text.replace(old_val_dataset, new_val_dataset, 1)
old_val_call = (
    '                use_pro_version=cfg.use_pro_version\n'
    '            )\n\n            # Add the loss value to the metrics\n'
)
new_val_call = (
    '                use_pro_version=cfg.use_pro_version,\n'
    '                cfg=cfg,\n'
    '            )\n\n            # Add the loss value to the metrics\n'
)
if old_val_call in text:
    text = text.replace(old_val_call, new_val_call, 1)
old_val_condition = (
    '            if cfg.use_val_set and log_step > 0 and log_step % cfg.val_freq == 0:\n'
)
new_val_condition = (
    '            if (cfg.use_val_set and log_step > 0 and log_step % cfg.val_freq == 0\n'
    '                    and (batch_idx + 1) % cfg.grad_accumulation_steps == 0):\n'
)
if old_val_condition in text:
    text = text.replace(old_val_condition, new_val_condition, 1)
old_val_mode = '    val_start_time = time.time()\n    vla.eval()\n'
new_val_mode = (
    '    val_start_time = time.time()\n'
    '    previous_phase = cfg.phase\n'
    '    cfg.phase = "Inference"\n'
    '    vla.eval()\n'
    '    if action_head is not None:\n'
    '        action_head.eval()\n'
    '    if proprio_projector is not None:\n'
    '        proprio_projector.eval()\n'
)
if old_val_mode in text:
    text = text.replace(old_val_mode, new_val_mode, 1)
old_val_end = (
    '    if distributed_state.is_main_process:\n'
    '        log_metrics_to_wandb(avg_val_metrics, "VLA Val", log_step, wandb)\n\n\n'
)
new_val_end = (
    '    if distributed_state.is_main_process:\n'
    '        log_metrics_to_wandb(avg_val_metrics, "VLA Val", log_step, wandb)\n\n'
    '    cfg.phase = previous_phase\n'
    '    if action_head is not None:\n'
    '        action_head.train()\n'
    '    if proprio_projector is not None:\n'
    '        proprio_projector.train()\n\n\n'
)
if old_val_end in text:
    text = text.replace(old_val_end, new_val_end, 1)
finetune.write_text(text)

# --- Resume support -------------------------------------------------------
# Upstream's --resume saves no optimizer or scheduler state and
# reads component checkpoints under a step-numbered name that
# --save_latest_checkpoint_only never writes. These patches add a
# self-contained --resume_checkpoint <run_dir> and leave --resume alone.
def replace_once(path, old, new, marker):
    text = path.read_text()
    if marker in text:
        return
    if old not in text:
        raise RuntimeError(f"Patch anchor not found in {path}")
    path.write_text(text.replace(old, new, 1))

replace_once(
    finetune,
    "        loss = torch.nn.L1Loss()(predicted_actions, ground_truth_actions)\n",
    "        if cfg.balance_z_loss:\n"
    "            per_timestep_l1 = torch.abs(\n"
    "                predicted_actions.float() - ground_truth_actions.float()\n"
    "            ).mean(dim=-1)\n"
    "            dz = ground_truth_actions[..., 2].float()\n"
    "            weights = torch.where(\n"
    "                dz > cfg.z_neutral_threshold, cfg.z_up_weight,\n"
    "                torch.where(dz < -cfg.z_neutral_threshold,\n"
    "                            cfg.z_down_weight, cfg.z_neutral_weight),\n"
    "            )\n"
    "            weights = weights / weights.mean().detach().clamp_min(1e-6)\n"
    "            loss = (per_timestep_l1 * weights).mean()\n"
    "        else:\n"
    "            loss = torch.nn.L1Loss()(predicted_actions, ground_truth_actions)\n",
    "per_timestep_l1 = torch.abs",
)

insert_after(
    finetune,
    "    z_up_weight: float = 1.8\n",
    '    resume_checkpoint: str = ""\n',
    "resume_checkpoint: str",
)
insert_after(
    finetune,
    '    resume_checkpoint: str = ""\n',
    "    resume_learning_rate: float = -1.0\n",
    "resume_learning_rate: float",
)

# The trained LoRA weights, which upstream's resume path drops on the floor.
replace_once(
    finetune,
    "        vla = get_peft_model(vla, lora_config)\n",
    "        if cfg.resume_checkpoint:\n"
    '            resume_adapter_dir = os.path.join(cfg.resume_checkpoint, "lora_adapter")\n'
    '            print(f"Resuming LoRA adapter from {resume_adapter_dir}")\n'
    "            vla = PeftModel.from_pretrained(vla, resume_adapter_dir, is_trainable=True)\n"
    "        else:\n"
    "            vla = get_peft_model(vla, lora_config)\n",
    "Resuming LoRA adapter from",
)

# Action head, proprio projector and (with FiLM) vision backbone. Component
# files are named either "<module>--latest_checkpoint.pt" or
# "<module>--<step>_checkpoint.pt" depending on save_latest_checkpoint_only.
replace_once(
    finetune,
    "    if cfg.resume:\n"
    "        state_dict = load_checkpoint(module_name, cfg.resum_vla_path, cfg.resume_step)\n"
    "        module.load_state_dict(state_dict)\n"
    "        print('loaded!!!!!!!!!')\n",
    "    if cfg.resume_checkpoint:\n"
    "        directory = Path(cfg.resume_checkpoint)\n"
    '        latest = directory / f"{module_name}--latest_checkpoint.pt"\n'
    "        if not latest.exists():\n"
    '            numbered = sorted(directory.glob(f"{module_name}--*_checkpoint.pt"),\n'
    "                              key=os.path.getmtime)\n"
    "            if not numbered:\n"
    '                raise FileNotFoundError(f"No {module_name} checkpoint in {directory}")\n'
    "            latest = numbered[-1]\n"
    '        print(f"Resuming {module_name} from {latest}")\n'
    "        module.load_state_dict(remove_ddp_in_checkpoint(\n"
    '            torch.load(latest, weights_only=True, map_location="cpu")))\n'
    "    elif cfg.resume:\n"
    "        state_dict = load_checkpoint(module_name, cfg.resum_vla_path, cfg.resume_step)\n"
    "        module.load_state_dict(state_dict)\n"
    "        print('loaded!!!!!!!!!')\n",
    "Resuming {module_name} from",
)

# `action_queries` is trained but lives outside the LoRA adapter, so
# PeftModel.save_pretrained drops it -- upstream only rescues it when merging.
insert_after(
    finetune,
    "    # Wrap VLA with DDP\n",
    "    if cfg.resume_checkpoint and cfg.use_lora:\n"
    '        extras_path = Path(cfg.resume_checkpoint) / "trainable_extras--latest_checkpoint.pt"\n'
    "        if extras_path.exists():\n"
    "            extras = remove_ddp_in_checkpoint(\n"
    '                torch.load(extras_path, weights_only=True, map_location="cpu"))\n'
    "            unexpected = vla.load_state_dict(extras, strict=False).unexpected_keys\n"
    "            if unexpected:\n"
    '                raise RuntimeError(f"Unexpected keys in {extras_path}: {unexpected}")\n'
    '            print(f"Resumed {len(extras)} trainable tensors outside the LoRA adapter")\n'
    "        else:\n"
    '            print(f"No {extras_path.name} to resume; action_queries start from the base model")\n',
    "trainable_extras--latest_checkpoint.pt",
)

# Optimizer moments and the LR schedule position. Without these a resumed run
# restarts AdamW cold and counts the num_steps_before_decay milestone from the
# resume point rather than from the start of training.
insert_after(
    finetune,
    "    scheduler = MultiStepLR(\n"
    "        optimizer,\n"
    "        milestones=[cfg.num_steps_before_decay],  # Number of steps after which LR will change\n"
    "        gamma=0.1,  # Multiplicative factor of learning rate decay\n"
    "    )\n",
    "\n"
    "    resume_step = 0\n"
    "    if cfg.resume_checkpoint:\n"
    '        training_state_path = Path(cfg.resume_checkpoint) / "training_state--latest_checkpoint.pt"\n'
    "        if not training_state_path.exists():\n"
    "            raise FileNotFoundError(\n"
    '                f"{training_state_path} is missing; that checkpoint predates resume support."\n'
    "            )\n"
    "        # weights_only=False: MultiStepLR's state holds a collections.Counter.\n"
    '        training_state = torch.load(training_state_path, map_location="cpu", weights_only=False)\n'
    '        expected_mode = "lora" if cfg.use_lora else "full"\n'
    '        if training_state.get("finetuning_mode", "lora") != expected_mode:\n'
    '            raise RuntimeError("Cannot resume across LoRA/full fine-tuning modes; start a fresh run.")\n'
    '        optimizer.load_state_dict(training_state["optimizer"])\n'
    '        scheduler.load_state_dict(training_state["scheduler"])\n'
    "        if cfg.resume_learning_rate > 0:\n"
    "            for param_group in optimizer.param_groups:\n"
    "                param_group[\"lr\"] = cfg.resume_learning_rate\n"
    "                param_group[\"initial_lr\"] = cfg.resume_learning_rate\n"
    "            scheduler.base_lrs = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    "            scheduler._last_lr = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    '            print(f"Overrode resumed learning rate to {cfg.resume_learning_rate:g}")\n'
    '        resume_step = training_state["step"]\n'
    '        print(f"Resuming optimizer and LR schedule at step {resume_step}")\n',
    "resume_step = 0",
)
# Add LR override support to runtimes patched by an older notebook version.
insert_after(
    finetune,
    '        scheduler.load_state_dict(training_state["scheduler"])\n',
    "        if cfg.resume_learning_rate > 0:\n"
    "            for param_group in optimizer.param_groups:\n"
    "                param_group[\"lr\"] = cfg.resume_learning_rate\n"
    "                param_group[\"initial_lr\"] = cfg.resume_learning_rate\n"
    "            scheduler.base_lrs = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    "            scheduler._last_lr = [cfg.resume_learning_rate] * len(optimizer.param_groups)\n"
    '            print(f"Overrode resumed learning rate to {cfg.resume_learning_rate:g}")\n',
    "Overrode resumed learning rate to",
)

# Steps stay absolute across a resume, so max_steps, save_freq and the W&B
# x-axis all mean the same thing they did in the original run.
replace_once(
    finetune,
    "            log_step = gradient_step_idx if not cfg.resume else cfg.resume_step + gradient_step_idx\n",
    "            log_step = resume_step + (\n"
    "                gradient_step_idx if not cfg.resume else cfg.resume_step + gradient_step_idx)\n",
    "log_step = resume_step + (",
)
replace_once(
    finetune,
    "with tqdm.tqdm(total=cfg.max_steps, leave=False) as progress:",
    "with tqdm.tqdm(total=cfg.max_steps, initial=resume_step, leave=False) as progress:",
    "initial=resume_step",
)

# Write the training state next to every checkpoint the trainer saves.
insert_after(
    finetune,
    "                    new_state_dict=RAW_STATE_DICT,\n"
    "                )\n",
    "                if distributed_state.is_main_process:\n"
    "                    state_dir = Path(run_dir) if cfg.save_latest_checkpoint_only \\\n"
    '                        else Path(str(run_dir) + f"--{log_step}_chkpt")\n'
    "                    torch.save(\n"
    '                        {"optimizer": optimizer.state_dict(),\n'
    '                         "scheduler": scheduler.state_dict(),\n'
    '                         "step": log_step,\n'
    '                         "finetuning_mode": "lora" if cfg.use_lora else "full"},\n'
    '                        state_dir / "training_state--latest_checkpoint.pt",\n'
    "                    )\n"
    "                    if cfg.use_lora:\n"
    "                        torch.save(\n"
    '                            {k: v for k, v in vla.state_dict().items() if "action_queries" in k},\n'
    '                            state_dir / "trainable_extras--latest_checkpoint.pt",\n'
    "                        )\n",
    '"training_state--latest_checkpoint.pt",',
)

# draccus turns an empty string on the command line into the literal "None",
# which is truthy: a fresh run would take the resume path and look for the
# adapter on the Hugging Face Hub. Normalise the optional flags before
# anything reads them.
insert_after(
    finetune,
    '    cfg.config_file_path = cfg.config_file_path.rstrip("/")\n',
    '    for _optional in ("resume_checkpoint", "wandb_entity"):\n'
    '        if getattr(cfg, _optional) in (None, "", "None", "none"):\n'
    '            setattr(cfg, _optional, "")\n',
    'for _optional in ("resume_checkpoint", "wandb_entity")',
)

# Full fine-tuning: unfreeze every backbone parameter and save/load a complete
# Hugging Face model, including action_queries. No PEFT wrapper is created.
replace_once(
    finetune,
    "    else:\n"
    "        for name, param in vla.named_parameters():\n"
    '            if "action_queries" in name:\n'
    "                param.requires_grad = True\n",
    "    else:\n"
    "        vla.requires_grad_(True)\n"
    "        total = sum(p.numel() for p in vla.parameters())\n"
    "        trainable = sum(p.numel() for p in vla.parameters() if p.requires_grad)\n"
    '        print(f"Full fine-tuning: {trainable:,}/{total:,} VLA parameters trainable")\n'
    "        assert trainable == total\n",
    "Full fine-tuning:",
)
replace_once(
    finetune,
    "    if cfg.use_minivlm:\n        hf_token = ''\n",
    "    if cfg.resume_checkpoint and not cfg.use_lora:\n"
    "        RAW_STATE_DICT = {}\n"
    "        vla = AutoModelForVision2Seq.from_pretrained(\n"
    "            cfg.resume_checkpoint, torch_dtype=torch.bfloat16,\n"
    "            low_cpu_mem_usage=False, trust_remote_code=False,\n"
    "        ).to(device_id)\n"
    '        print(f"Resumed full VLA from {cfg.resume_checkpoint}")\n'
    "    elif cfg.use_minivlm:\n        hf_token = ''\n",
    "Resumed full VLA from",
)
replace_once(
    finetune,
    "        if cfg.use_fz:\n"
    "            vla.module.save_pretrained(checkpoint_dir) # directly save checkpoint without lora\n",
    "        if not cfg.use_lora:\n"
    "            vla.module.save_pretrained(checkpoint_dir)  # Complete trainable VLA\n",
    "# Complete trainable VLA",
)
replace_once(
    finetune,
    "        os.makedirs(adapter_dir, exist_ok=True)\n",
    "        if cfg.use_lora:\n            os.makedirs(adapter_dir, exist_ok=True)\n",
    "if cfg.use_lora:\n            os.makedirs(adapter_dir",
)
# The source VLM and remapped weights are needed only for initialization when
# full fine-tuning; retaining them wastes host memory for the entire run.
replace_once(
    finetune,
    "        del old_state_dict\n",
    "        del old_state_dict\n"
    "        if not cfg.use_lora:\n"
    "            del vlm\n"
    "            RAW_STATE_DICT = {}  # No LoRA merge needs the original weights\n",
    "# No LoRA merge needs the original weights",
)

# Upgrade resume patches already installed by an earlier LoRA notebook.
insert_after(
    finetune,
    '        training_state = torch.load(training_state_path, map_location="cpu", weights_only=False)\n',
    '        expected_mode = "lora" if cfg.use_lora else "full"\n'
    '        if training_state.get("finetuning_mode", "lora") != expected_mode:\n'
    '            raise RuntimeError("Cannot resume across LoRA/full fine-tuning modes; start a fresh run.")\n',
    'expected_mode = "lora"',
)
text = finetune.read_text()
text = text.replace(
    '    if cfg.resume_checkpoint:\n        extras_path =',
    '    if cfg.resume_checkpoint and cfg.use_lora:\n        extras_path =',
    1,
)
text = text.replace(
    '                         "step": log_step},\n',
    '                         "step": log_step,\n'
    '                         "finetuning_mode": "lora" if cfg.use_lora else "full"},\n',
    1,
)
old_extras_save = (
    '                    torch.save(\n'
    '                        {k: v for k, v in vla.state_dict().items() if "action_queries" in k},\n'
    '                        state_dir / "trainable_extras--latest_checkpoint.pt",\n'
    '                    )\n'
)
text = text.replace(
    old_extras_save,
    '                    if cfg.use_lora:\n' + ''.join(
        '    ' + line for line in old_extras_save.splitlines(keepends=True)),
    1,
)
finetune.write_text(text)

print("RobotArmLearning full fine-tuning setup is ready.")


## 3. Download and validate demonstrations

Only `data.npz` and `meta.json` are fetched. Re-running this cell resumes from the Hugging Face cache.

In [ ]:
download_program = f'''
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id={DATASET_REPO!r},
    repo_type="dataset",
    revision={DATASET_REVISION!r},
    local_dir={str(CODE_DIR)!r},
    allow_patterns=["data/episode_*/data.npz", "data/episode_*/meta.json"],
)
'''
run([PYTHON, "-c", download_program])

validate_program = f'''
from pathlib import Path
import numpy as np
root = Path({str(CODE_DIR / 'data')!r})
episodes = sorted(root.glob("episode_*"))
required = {{"t", "q", "ctrl", "ee_pos", "ee_quat", "obj_pos", "obj_quat", "gripper"}}
if not episodes:
    raise RuntimeError("No episodes downloaded")
for episode in episodes:
    with np.load(episode / "data.npz") as data:
        missing = required - set(data.files)
        if missing:
            raise RuntimeError(f"{{episode.name}} is missing {{sorted(missing)}}")
print(f"Validated {{len(episodes)}} episodes")
'''
run([PYTHON, "-c", validate_program])


## 4. Render shoulder/wrist observations and build RLDS

The next cell makes sure MuJoCo renders on the GPU. Colab ships the NVIDIA
driver but usually omits the libglvnd vendor file that advertises it to EGL,
and MuJoCo then falls back to Mesa's `llvmpipe` software rasteriser without
warning. The images come out correct, roughly a hundred times slower --
minutes of rendering become hours. The check is a hard failure rather than a
warning for that reason.

Rendering is resumable episode by episode. RLDS creation deterministically holds out
complete episodes for validation, avoiding leakage between neighboring frames. The
generated JPEGs and TFRecords remain on the ephemeral Colab disk; only checkpoints
need to be preserved.

In [ ]:
# Headless GPU rendering, verified rather than assumed.
NVIDIA_ICD = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not NVIDIA_ICD.exists():
    NVIDIA_ICD.parent.mkdir(parents=True, exist_ok=True)
    NVIDIA_ICD.write_text('{"file_format_version": "1.0.0", '
                          '"ICD": {"library_path": "libEGL_nvidia.so.0"}}')
    print("wrote", NVIDIA_ICD)

GL_PROBE = """
import os
os.environ["MUJOCO_GL"] = "egl"
from mujoco.egl import GLContext
from OpenGL import GL
context = GLContext(64, 64)  # Bound to a name: a temporary is freed
context.make_current()       # before glGetString can be answered.
print("renderer", GL.glGetString(GL.GL_RENDERER).decode())
"""
SOFTWARE_GL = ("llvmpipe", "softpipe", "swrast")

def gl_renderer(env):
    """The OpenGL renderer MuJoCo's EGL context lands on, or None if it fails."""
    probe = subprocess.run([PYTHON, "-c", GL_PROBE], env=env,
                           capture_output=True, text=True)
    for line in probe.stdout.splitlines():
        if line.startswith("renderer "):
            return line[len("renderer "):]
    return None

def is_software(name):
    return name is None or any(s in name.lower() for s in SOFTWARE_GL)

render_env = os.environ.copy()
render_env["MUJOCO_GL"] = "egl"
renderer = gl_renderer(render_env)
if is_software(renderer):
    # mujoco.egl takes the first device that initialises, which may be Mesa's
    # even once the NVIDIA one is enumerated. Select a device explicitly.
    for device_id in range(4):
        candidate = dict(render_env, MUJOCO_EGL_DEVICE_ID=str(device_id))
        if not is_software(gl_renderer(candidate)):
            render_env = candidate
            renderer = gl_renderer(candidate)
            break
if is_software(renderer):
    raise RuntimeError(
        f"MuJoCo EGL resolved to {renderer!r}, not a GPU. Rendering would take "
        "hours. Confirm the driver's EGL library is installed -- "
        "`!ldconfig -p | grep libEGL_nvidia` should print a path; if it does "
        "not, `!apt-get install -y -qq libnvidia-gl-<version>` matching "
        "nvidia-smi, then rerun this cell.")
print("EGL renderer:", renderer,
      "(device", render_env.get("MUJOCO_EGL_DEVICE_ID", "auto") + ")")


In [ ]:
RENDERED_DIR = VLA_DIR / "data/robot_arm_learning_rendered"
RLDS_DIR = VLA_DIR / "data/robot_arm_learning"
run([PYTHON, CODE_DIR / "teleop/render_vla_dataset.py",
     "--input", CODE_DIR / "data",
     "--output", RENDERED_DIR,
     "--hz", str(SAMPLE_HZ), "--size", "256"], env=render_env)
run([PYTHON, CODE_DIR / "teleop/build_robot_arm_learning_rlds.py",
     "--rendered-dir", RENDERED_DIR,
     "--data-dir", RLDS_DIR,
     "--instruction", INSTRUCTION,
     "--val-fraction", str(VAL_FRACTION),
     "--split-seed", str(VAL_SPLIT_SEED)])


In [ ]:
# Sanity-check one rendered observation pair.
from IPython.display import display, Image
sample = sorted(RENDERED_DIR.glob("episode_*"))[0]
print(sample.name)
display(Image(filename=str(sample / "shoulder/00000.jpg"), width=320))
display(Image(filename=str(sample / "wrist/00000.jpg"), width=320))


## 5. Download the base model

In [ ]:
MODEL_DIR = VLA_DIR / "pretrained_models/prism-qwen25-extra-dinosiglip-224px-0_5b"
model_program = f'''
from huggingface_hub import snapshot_download
snapshot_download(repo_id={MODEL_REPO!r}, local_dir={str(MODEL_DIR)!r})
'''
run([PYTHON, "-c", model_program])


## 6. Fine-tune

Full fine-tuning updates every VLA parameter, including both vision backbones, the language model, multimodal projector, and action queries, plus the action head and proprio projector. LoRA is disabled. The defaults use an effective batch size of 8 (`batch_size=1`, eight accumulation steps), language-model gradient checkpointing, and `LEARNING_RATE=2e-5`. Start with an A100; memory use has not been benchmarked for this configuration. Every `VAL_FREQ` optimizer steps, evaluation runs without image augmentation on the held-out episodes and logs `VLA Val/Loss`, `VLA Val/Current Action L1 Loss`, and `VLA Val/Next Actions L1 Loss`. The last metric is the direct counterpart to the training curve. `VAL_TIME_LIMIT` bounds the added runtime. If CUDA reports an out-of-memory error at batch size 1, use a GPU with more memory; increasing accumulation alone does not reduce memory at that point. Set `WANDB_ENTITY` in the settings cell to log to wandb.ai; leaving it empty keeps runs offline in `VLA-Adapter/wandb/`. An online run needs an API key from <https://wandb.ai/authorize>, taken from the `WANDB_API_KEY` Colab secret (key icon in the sidebar, with notebook access enabled) or prompted for if that secret is missing. On an Ampere-or-newer GPU, `uv pip install --python PYTHON flash_attn==2.5.5` is picked up automatically by the attention patch and speeds up training; it is optional. With `SAVE_TO_DRIVE=False`, run the export cell before the Colab runtime disconnects.

This cell prints its `RUN_ID`. A checkpoint trained before this validation split existed has already seen every episode, so it cannot produce an honest held-out loss; leave `RESUME_RUN_ID` empty and start one fresh run. After that, interrupted split-aware runs can be continued by setting `RESUME_RUN_ID` to their value and rerunning from the top: training restores the complete VLA model (including action queries), action head, proprio projector, AdamW moments and LR schedule position from the last checkpoint, and step numbers stay absolute, so `MAX_STEPS` still means total steps. The notebook records and checks the split configuration before allowing a resume. Only full-finetuning checkpoints from this notebook can be resumed; old LoRA runs require a fresh `RUN_ID`. Resume requires the full model weights, `config.json`, component checkpoints, and `training_state--latest_checkpoint.pt`. The RLDS input pipeline restarts at the head of its shuffle stream rather than mid-epoch.

In [ ]:
from datetime import datetime, timezone
import json

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")  # Already mounted above; returns immediately.
    OUTPUT_ROOT = Path(DRIVE_OUTPUT)
else:
    OUTPUT_ROOT = VLA_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = RESUME_RUN_ID or (
    "robot-arm-learning-full-colab-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
RESUME_DIR = OUTPUT_ROOT / RUN_ID if RESUME_RUN_ID else None
def require_full_checkpoint(directory):
    required = ["config.json", "training_state--latest_checkpoint.pt",
                "action_head--latest_checkpoint.pt", "proprio_projector--latest_checkpoint.pt"]
    missing = [name for name in required if not (directory / name).is_file()]
    has_weights = any((directory / name).is_file() for name in (
        "model.safetensors", "model.safetensors.index.json",
        "pytorch_model.bin", "pytorch_model.bin.index.json"))
    if missing or not has_weights:
        raise RuntimeError(
            f"No complete full-finetuning checkpoint at {directory}. Missing: {missing}; "
            f"full model weights present: {has_weights}. Old LoRA runs cannot be resumed; "
            "leave RESUME_RUN_ID empty to start fresh.")

if RESUME_DIR is not None:
    require_full_checkpoint(RESUME_DIR)
validation_config = {
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "rlds_version": "1.1.0",
    "val_fraction": VAL_FRACTION,
    "split_seed": VAL_SPLIT_SEED,
}
validation_config_path = OUTPUT_ROOT / RUN_ID / "validation_split.json"
if RESUME_DIR is not None:
    if not validation_config_path.exists():
        raise RuntimeError(
            "This run predates the held-out validation split and has seen all episodes. "
            "Leave RESUME_RUN_ID empty and start a fresh run for valid metrics.")
    saved_validation_config = json.loads(validation_config_path.read_text())
    if saved_validation_config != validation_config:
        raise RuntimeError(
            f"Validation split changed since this run began: "
            f"{saved_validation_config} != {validation_config}")
else:
    validation_config_path.parent.mkdir(parents=True, exist_ok=True)
    validation_config_path.write_text(json.dumps(validation_config, indent=2) + "\n")
print("RUN_ID:", RUN_ID, "(resuming)" if RESUME_DIR else "(new run)")

WANDB_MODE = "online" if WANDB_ENTITY else "offline"
if WANDB_ENTITY and not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        import getpass
        os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key (wandb.ai/authorize): ")

command = [
    VENV / "bin/torchrun", "--standalone", "--nnodes", "1", "--nproc-per-node", "1",
    VLA_DIR / "vla-scripts/finetune.py",
    "--vlm_path", MODEL_DIR,
    "--config_file_path", VLA_DIR / "pretrained_models/configs",
    "--data_root_dir", RLDS_DIR,
    "--dataset_name", "robot_arm_learning_panthera",
    "--run_root_dir", OUTPUT_ROOT,
    "--run_id_override", RUN_ID,
    "--use_film", "False",
    "--num_images_in_input", "2",
    "--use_proprio", "True",
    "--use_lora", "False",
    "--use_fz", "True",
    "--use_minivlm", "True",
    "--image_aug", "True",
    "--shuffle_buffer_size", "12000",
    "--use_val_set", "True",
    "--val_freq", str(VAL_FREQ),
    "--val_time_limit", str(VAL_TIME_LIMIT),
    "--num_steps_before_decay", str(max(1, int(MAX_STEPS * 0.8))),
    "--max_steps", str(MAX_STEPS),
    "--save_freq", str(SAVE_FREQ),
    "--save_latest_checkpoint_only", "True",
    "--batch_size", str(BATCH_SIZE),
    "--grad_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--learning_rate", str(LEARNING_RATE),
    "--use_pro_version", "True",
    "--use_gradient_checkpointing", "True",
    "--balance_z_loss", "True",
    # Left at its 0.1 default, the warmup block rewrites the learning rate back
    # to --learning_rate on every step, which silently cancels the 10x decay at
    # --num_steps_before_decay. 0 disables it and leaves the scheduler in charge.
    "--lr_warmup_steps", "0",
    "--wandb_project", WANDB_PROJECT,
]
# Only when set: draccus reads an empty value as the string "None".
if WANDB_ENTITY:
    command += ["--wandb_entity", WANDB_ENTITY]
if RESUME_DIR:
    command += ["--resume_checkpoint", str(RESUME_DIR)]
    if RESUME_LEARNING_RATE is not None:
        command += ["--resume_learning_rate", str(RESUME_LEARNING_RATE)]
train_env = os.environ.copy()
train_env.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "WANDB_MODE": WANDB_MODE,
    "WANDB_RUN_ID": RUN_ID,
    "WANDB_RESUME": "allow",
    "PYTHONPATH": str(VLA_DIR),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})
run(command, cwd=VLA_DIR, env=train_env)


## 7. Package the fully fine-tuned model

This archive contains the full VLA weights and config, action head, proprio projector, processor files, action-normalization statistics, and optimizer/scheduler state for resuming. The full model includes the trained vision and language backbones and action queries; no separate base-model weights or LoRA merge are needed. Full checkpoints and their archives are substantially larger than LoRA adapters. The repository's current `rollout.py` loader still expects LoRA artifacts and will need full-checkpoint loading support before it can run this model.

For an unattended overnight run, leave `SAVE_TO_DRIVE = True` and set `SHUTDOWN_WHEN_DONE = True`: the archive is copied to Drive and the last cell releases the runtime instead of letting it idle until Colab reclaims it. Use **Runtime -> Run all** and keep the browser tab open unless your Colab plan includes background execution.

In [ ]:
import tarfile

RUN_DIR = OUTPUT_ROOT / RUN_ID
require_full_checkpoint(RUN_DIR)
ARCHIVE = ROOT / f"{RUN_ID}.tar.gz"
with tarfile.open(ARCHIVE, "w:gz") as archive:
    archive.add(RUN_DIR, arcname=RUN_ID)
print(f"Created {ARCHIVE} ({ARCHIVE.stat().st_size / 2**20:.1f} MiB)")

if SAVE_TO_DRIVE:
    destination = Path(DRIVE_OUTPUT) / ARCHIVE.name
    shutil.copy2(ARCHIVE, destination)
    print("Copied to", destination)
else:
    from google.colab import files
    files.download(str(ARCHIVE))


In [ ]:
# Release the runtime, so an overnight run stops billing when it finishes
# rather than when Colab's idle timeout notices. Everything on the local disk
# goes with it -- this cell is deliberately last, after the export above.
#
# A failing cell leaves the runtime up: Colab abandons the rest of a queued
# run on the first error, so this never fires and the traceback survives for
# as long as the idle timeout allows.
if SHUTDOWN_WHEN_DONE:
    if not SAVE_TO_DRIVE:
        raise RuntimeError(
            "SAVE_TO_DRIVE is False: checkpoints live on the runtime disk and "
            "would be destroyed. Save them before shutting down."
        )
    print("Archive at", destination)
    from google.colab import runtime
    runtime.unassign()
